# 03 — Business Impact & ROI Analysis
Translating model outputs into clinical and financial value for a value-based care practice.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.append("../src")
from roi_calculator import (compute_panel_roi, quality_gap_value_table,
                             sdoh_risk_summary, format_roi_report)

df = pd.read_csv("../data/raw/patient_panel_flat.csv")
scored = pd.read_csv("../data/processed/scored_panel.csv")


## Panel statistics

In [ ]:
n_high_critical = df["outreach_priority"].isin(["Critical", "High"]).sum()
avg_gaps = df[df["outreach_priority"].isin(["Critical", "High"])]["n_quality_gaps"].mean()
er_visits = int(df["er_visits_12m"].sum())
hosp = int(df["hospitalizations_12m"].sum())

print(f"Panel size:                {len(df):,}")
print(f"High/Critical patients:    {n_high_critical} ({n_high_critical/len(df)*100:.0f}%)")
print(f"Avg quality gaps (H/C):    {avg_gaps:.1f}")
print(f"Total ER visits (12mo):    {er_visits}")
print(f"Total hospitalizations:    {hosp}")


## ROI calculation

In [ ]:
roi = compute_panel_roi(
    panel_size=len(df),
    n_high_risk=n_high_critical,
    n_outreached=int(n_high_critical * 0.65),
    avg_quality_gaps_per_patient=avg_gaps,
    er_visits_baseline=er_visits,
    hospitalizations_baseline=hosp,
)
print(format_roi_report(roi))


## Quality gap value table

In [ ]:
gap_table = quality_gap_value_table(df)
print("Quality gap closure — estimated value per closure:")
print(gap_table.to_string(index=False))


## ROI waterfall chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

categories = ["Quality\nbonus", "Hospital\ncost avoided", "ER cost\navoided",
              "Program\ncost", "Net\nbenefit"]
values = [roi["quality_bonus_value"], roi["hosp_cost_avoided"],
          roi["er_cost_avoided"], -roi["outreach_program_cost"],
          roi["net_annual_benefit"]]
bar_colors = ["#22c55e", "#22c55e", "#22c55e", "#ef4444", "#3b82f6"]

bars = ax.bar(categories, values, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("USD (annual)")
ax.set_title("Annual value from proactive panel management — 800-patient panel", fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))

for bar, val in zip(bars, values):
    offset = 3000 if val >= 0 else -14000
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
            f"${abs(val)/1e3:.0f}K", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/roi_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()


## SDOH summary by priority tier

In [ ]:
sdoh_summary = sdoh_risk_summary(df)
print(sdoh_summary.to_string(index=False))
print("")
print("Key insight: SDOH barriers cluster in High/Critical tiers.")
print("Outreach strategy must account for non-clinical access barriers")
print("(transportation vouchers, food assistance referrals, telehealth options).")


## Why this matters for value-based care

The financial case is straightforward: proactive outreach costs less than reactive care. A phone call to close an HbA1c gap costs $45. An avoidable hospitalization for uncontrolled diabetes costs $12,000.

But the more important number is the one that doesn't appear in this notebook: the patient who gets their HbA1c checked in the clinic instead of in the ICU. The model exists to find that patient before the crisis — which is the entire premise of value-based care.

**For Counterpart Health specifically:** This model operationalizes the same workflow that Counterpart Assistant supports for PCPs — identifying which patients in a chronic disease panel need attention today, before an acute event forces the issue.
